# Étape 5 : comparer les stratégies à risque égal, leur coût et le risque qui reste

Les critères de ce carnet sont ceux de la section VIII de `research/plan_projet.md`, écrits le 28 septembre 2026 avec la phase 0 de l'étape 4, donc avant tout résultat de couverture. Les séries des stratégies viennent de `data/processed/couverture_valeurs.csv.gz`, produit par `src/couvrir_diversifier.ipynb`.

Le principe : une stratégie qui réduit le risque ne fait rien d'utile si une simple vente d'actions contre des liquidités obtient le même risque avec au moins le même rendement. Pour chaque stratégie, je construis donc le mélange de la série non couverte et de liquidités qui a exactement le même risque, puis je compare les rendements.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src import risque
from src import couverture as cv
from src.collecter_couverture import verifier
from src.empreintes import correspond, empreinte

SORTIE = RACINE / "data" / "processed"
BRUT = RACINE / "data" / "raw"
verifier(RACINE)
manifeste4 = json.loads((SORTIE / "couverture_manifest.json").read_text(encoding="utf-8"))
for nom, sha in manifeste4["sorties_sha256"].items():
    if not correspond(SORTIE / nom, sha):
        raise ValueError(f"Sortie de l'etape 4 modifiee depuis son manifeste : {nom}")

table = pd.read_csv(SORTIE / "couverture_valeurs.csv.gz", index_col=0)
table.index = table.index.astype(str)
dates = table.index
comparaisons = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)
taux = pd.read_csv(BRUT / "taux_sans_risque.csv", index_col=0)["taux_annuel_pct"].reindex(dates).ffill()
sans_risque = (1 + taux / 100) ** (1 / 252) - 1
r_spy = comparaisons["SPY"].pct_change()
annuel = cv.premieres_seances_annee(dates)
episodes = pd.read_csv(SORTIE / "episodes_tension.csv")
tension = pd.Series(False, index=dates)
for e in episodes.itertuples():
    tension.loc[e.debut:e.creux] = True

SERIES = manifeste4["series"]
# Les melanges de liquidites sont leurs propres jumeaux : ils servent de reference, pas de candidats.
STRATEGIES = [s for s in manifeste4["strategies"] if s != "non_couvert" and not s.startswith("liquidites")]
print(table.shape, len(SERIES), "series,", len(STRATEGIES), "strategies")

(6709, 271) 21 series, 10 strategies


### La variante de prix d'option calibrée

L'étape 4 a montré que le modèle surévalue les puts à la monnaie d'environ 3,8 points de volatilité. Les deux stratégies de puts sont donc aussi comparées dans leur variante à trois points sous le `VIX`, reconstruite ici avec les mêmes fonctions. Elles apparaissent sous les noms `put_90_calibre` et `put_95_calibre` ; la base de la phase 0 reste dans les tableaux.

In [2]:
def lire_indice(chemin):
    df = pd.read_csv(chemin)
    df.index = pd.Index(df.date.str[:10], name="date")
    return df

gspc = lire_indice(BRUT / "benchmarks" / "GSPC.csv")["Close"].reindex(dates)
spy_brut = lire_indice(BRUT / "benchmarks" / "SPY.csv")
div = spy_brut["Dividends"].copy()
div.index = pd.to_datetime(div.index)
rdt_div = div.rolling("365D").sum() / spy_brut["Close"].set_axis(div.index)
rdt_div.index = rdt_div.index.strftime("%Y-%m-%d")
q = np.log1p(rdt_div.reindex(dates).ffill())
vix = pd.read_csv(BRUT / "couverture" / "VIX.csv", index_col=0)["Close"].reindex(dates) / 100
taux_continu = np.log1p(taux / 100)
fins = cv.fins_de_mois(dates)

calibres = {}
for serie in SERIES:
    r = table[f"{serie}|non_couvert"].pct_change()
    beta = cv.beta_glissant(r, r_spy)
    for niveau_k in (0.90, 0.95):
        v, _, _ = cv.protection_put(r, gspc, vix - 0.03, taux_continu, q, beta, fins, niveau_k)
        calibres[f"{serie}|put_{round(100 * niveau_k)}_calibre"] = v
table = pd.concat([table, pd.DataFrame(calibres)], axis=1)
STRATEGIES += ["put_90_calibre", "put_95_calibre"]
print(table.shape)

(6709, 313)


## 1. La valeur ajoutée à risque égal

Deux mesures de risque, fixées dans la section VIII : la volatilité quotidienne sur la période complète et la perte moyenne au-delà de la VaR à 99 %. Pour chacune, la part de la série dans le mélange de liquidités est cherchée par dichotomie jusqu'à égaliser le risque de la stratégie. La valeur ajoutée est l'écart de rendement annualisé géométrique entre la stratégie et ce mélange.

In [3]:
MESURES = {"volatilite": risque.volatilite, "es99": lambda r: risque.perte_au_dela(r, 0.99)}

lignes, jumeaux = [], {}
for serie in SERIES:
    r_s = table[f"{serie}|non_couvert"].pct_change()
    for strat in STRATEGIES:
        colonne = f"{serie}|{strat}"
        if colonne not in table:
            continue
        v = table[colonne]
        r_v = v.pct_change().dropna()
        for nom, mesure in MESURES.items():
            part, jumeau = cv.part_a_risque_egal(r_s, sans_risque, annuel, mesure(r_v), mesure)
            jumeaux[(serie, strat, nom)] = jumeau
            lignes.append({"serie": serie, "strategie": strat, "risque": nom, "part_serie_jumeau": part,
                           "risque_strategie": mesure(r_v),
                           "risque_jumeau": mesure(jumeau.pct_change().dropna()),
                           "rendement_strategie": risque.annualiser_rendement(v),
                           "rendement_jumeau": risque.annualiser_rendement(jumeau)})
egal = pd.DataFrame(lignes)
egal["valeur_ajoutee"] = egal.rendement_strategie - egal.rendement_jumeau
print("ecart maximal de risque entre strategie et jumeau : %.2e"
      % (egal.risque_strategie - egal.risque_jumeau).abs().max())
for serie in ["P1_reeq", "P1_cons", "T1_reeq"]:
    print("===", serie)
    print(egal[egal.serie == serie].pivot(index="strategie", columns="risque",
          values=["part_serie_jumeau", "valeur_ajoutee"]).round(4).to_string())

ecart maximal de risque entre strategie et jumeau : 1.07e-08
=== P1_reeq
               part_serie_jumeau            valeur_ajoutee           
risque                      es99 volatilite           es99 volatilite
strategie                                                            
futur_100                 0.4073     0.3906         0.0118     0.0147
futur_50                  0.6144     0.6068         0.0214     0.0226
hors_theme_60             0.9732     0.9308        -0.0052     0.0011
hors_theme_80             0.9835     0.9629        -0.0019     0.0011
put_90                    0.7251     0.8480         0.0126    -0.0066
put_90_calibre            0.7411     0.8674         0.0292     0.0095
put_95                    0.6333     0.7269        -0.0252    -0.0402
put_95_calibre            0.6493     0.7566         0.0334     0.0162
treso_inter_60            0.5700     0.5792         0.0145     0.0129
treso_inter_80            0.7806     0.7860         0.0080     0.0072
treso_long_60    

## 2. La robustesse exigée : les deux moitiés et le bootstrap

Les moitiés sont coupées au 1er mai 2013. Le jumeau est celui de la période complète ; seule la fenêtre de mesure change. Le bootstrap par blocs d'un trimestre, 2 000 tirages, porte sur l'écart des rendements quotidiens moyens annualisés entre la stratégie et son jumeau, aux mêmes dates. C'est une moyenne arithmétique, utilisée comme statistique de test et jamais présentée comme un rendement. Une stratégie n'est dite supérieure à son jumeau que si les deux moitiés donnent le même signe et si l'intervalle exclut zéro.

In [4]:
COUPURE = "2013-05-01"
BLOC, TIRAGES = 63, 2000
moyenne_annuelle = lambda x: x.mean() * 252

lignes = []
for ligne in egal.itertuples():
    v = table[f"{ligne.serie}|{ligne.strategie}"]
    j = jumeaux[(ligne.serie, ligne.strategie, ligne.risque)]
    moities = []
    for fenetre in [slice(None, COUPURE), slice(COUPURE, None)]:
        moities.append(risque.annualiser_rendement(v.loc[fenetre]) - risque.annualiser_rendement(j.loc[fenetre]))
    a, b = v.pct_change().dropna().to_numpy(), j.pct_change().dropna().to_numpy()
    test = risque.ecart_bootstrap(a, b, moyenne_annuelle, BLOC, TIRAGES)
    lignes.append({"premiere_moitie": moities[0], "seconde_moitie": moities[1],
                   "ecart_moyen": test["ecart"], "bas": test["bas"], "haut": test["haut"],
                   "p_bilaterale": test["p_bilaterale"]})
egal = pd.concat([egal, pd.DataFrame(lignes, index=egal.index)], axis=1)
meme_signe = np.sign(egal.premiere_moitie) == np.sign(egal.seconde_moitie)
exclut_zero = (egal.bas > 0) | (egal.haut < 0)
egal["verdict"] = np.where(meme_signe & exclut_zero,
                           np.where(egal.valeur_ajoutee > 0, "superieure", "inferieure"), "indecidable")
egal.to_csv(SORTIE / "strategies_risque_egal.csv", index=False, encoding="utf-8")

print(egal[egal.serie.isin(["P1_reeq", "P1_cons"])][["serie", "strategie", "risque", "valeur_ajoutee",
      "premiere_moitie", "seconde_moitie", "bas", "haut", "verdict"]].round(4).to_string(index=False))
print()
vingt = egal[egal.serie.str.startswith("P")]
decompte = vingt.groupby(["risque", "strategie"]).verdict.value_counts().unstack(fill_value=0)
decompte["mediane_valeur_ajoutee"] = vingt.groupby(["risque", "strategie"]).valeur_ajoutee.median()
print(decompte.round(4).to_string())

  serie      strategie     risque  valeur_ajoutee  premiere_moitie  seconde_moitie     bas    haut     verdict
P1_reeq treso_inter_80 volatilite          0.0072           0.0124          0.0017  0.0027  0.0096  superieure
P1_reeq treso_inter_80       es99          0.0080           0.0129          0.0029  0.0037  0.0104  superieure
P1_reeq treso_inter_60 volatilite          0.0129           0.0233          0.0022  0.0046  0.0183  superieure
P1_reeq treso_inter_60       es99          0.0145           0.0244          0.0042  0.0063  0.0196  superieure
P1_reeq  treso_long_80 volatilite          0.0116           0.0200          0.0028  0.0020  0.0183  superieure
P1_reeq  treso_long_80       es99          0.0123           0.0204          0.0037  0.0027  0.0191  superieure
P1_reeq  treso_long_60 volatilite          0.0168           0.0348         -0.0017 -0.0015  0.0320 indecidable
P1_reeq  treso_long_60       es99          0.0180           0.0357         -0.0001 -0.0001  0.0332 indecidable
P

## 3. Le coût

Pour chaque stratégie, sans les convertir en un classement unique : le coût explicite annuel, le rendement abandonné, et les réductions de volatilité, de repli maximal et de perte au-delà de la VaR à 99 %, chacune rapportée au rendement abandonné quand celui-ci est positif.

In [5]:
mesures4 = pd.read_csv(SORTIE / "couverture_mesures.csv")
sens = pd.read_csv(SORTIE / "couverture_sensibilite_puts.csv")
cout = mesures4[mesures4.strategie != "non_couvert"][["serie", "strategie", "cout_explicite", "rendement_abandonne",
       "reduction_vol_quotidienne", "reduction_repli_max", "reduction_es99"]].copy()
for col in ["reduction_vol_quotidienne", "reduction_repli_max", "reduction_es99"]:
    cout[f"{col}_par_point"] = np.where(cout.rendement_abandonne > 0,
                                        cout[col] / (100 * cout.rendement_abandonne), np.nan)
cout.to_csv(SORTIE / "strategies_cout.csv", index=False, encoding="utf-8")
print(cout[cout.serie == "P1_reeq"].set_index("strategie").round(4).to_string())
print()
print("medianes sur les vingt series :")
print(cout[cout.serie.str.startswith("P")].groupby("strategie").median(numeric_only=True).round(4).to_string())

                  serie  cout_explicite  rendement_abandonne  reduction_vol_quotidienne  reduction_repli_max  reduction_es99  reduction_vol_quotidienne_par_point  reduction_repli_max_par_point  reduction_es99_par_point
strategie                                                                                                                                                                                                                 
liquidites_80   P1_reeq          0.0001               0.0303                     0.2090               0.1767          0.2270                               0.0690                         0.0583                    0.0749
liquidites_60   P1_reeq          0.0001               0.0622                     0.4100               0.3617          0.4346                               0.0659                         0.0581                    0.0698
treso_inter_80  P1_reeq          0.0001               0.0253                     0.2233               0.2019          0.2479

## 4. Le risque qui reste

Pour chaque stratégie appliquée à `P1`, trois mesures fixées dans la section VIII : la part de la variance que `SPY` n'explique pas, la pire perte parmi les six épisodes, et la perte moyenne au-delà de la VaR à 99 % calculée sur les seules séances de tension.

In [6]:
lignes = []
for serie in SERIES:
    for strat in ["non_couvert", "liquidites_80", "liquidites_60"] + STRATEGIES:
        colonne = f"{serie}|{strat}"
        if colonne not in table:
            continue
        v = table[colonne]
        r = v.pct_change().dropna()
        m = r_spy.reindex(r.index)
        ok = m.notna()
        r2 = np.corrcoef(r[ok], m[ok])[0, 1] ** 2
        pires = {e.debut[:4]: v.loc[e.creux] / v.loc[e.debut] - 1 for e in episodes.itertuples()}
        pire = min(pires, key=pires.get)
        lignes.append({"serie": serie, "strategie": strat, "part_hors_marche": 1 - r2,
                       "vol_hors_marche": risque.volatilite(r) * np.sqrt(1 - r2),
                       "pire_episode": pire, "perte_pire_episode": pires[pire],
                       "es99_tension": risque.perte_au_dela(r[tension.reindex(r.index)], 0.99)})
residuel = pd.DataFrame(lignes)
residuel.to_csv(SORTIE / "strategies_risque_residuel.csv", index=False, encoding="utf-8")
for serie in ["P1_reeq", "P1_cons"]:
    print("===", serie)
    print(residuel[residuel.serie == serie].set_index("strategie").drop(columns="serie").round(4).to_string())

=== P1_reeq
                part_hors_marche  vol_hors_marche pire_episode  perte_pire_episode  es99_tension
strategie                                                                                       
non_couvert               0.1481           0.0836         2007             -0.4999        0.0768
liquidites_80             0.1595           0.0686         2007             -0.4113        0.0581
liquidites_60             0.1731           0.0533         2007             -0.3151        0.0418
treso_inter_80            0.1678           0.0691         2007             -0.3914        0.0565
treso_inter_60            0.2086           0.0565         2007             -0.2715        0.0393
treso_long_80             0.1917           0.0725         2007             -0.3854        0.0552
treso_long_60             0.3363           0.0715         2007             -0.2592        0.0388
hors_theme_80             0.1198           0.0722         2007             -0.5027        0.0759
hors_theme_60     

### Le risque résiduel de la couverture par contrats à terme, par maillon

Même méthode qu'à la tâche 19 de l'étape 3 : poids du dernier relevé mensuel, covariance des 252 dernières séances, contributions d'Euler qui somment à la volatilité. La vente de contrats à terme entre comme une ligne supplémentaire, de poids égal à moins le bêta estimé sur la même fenêtre. La lecture décrit donc l'état au 4 septembre 2026, avec la même limite qu'à l'étape 3.

In [7]:
appartenance = pd.read_csv(SORTIE / "appartenance.csv")
rendements = pd.read_csv(SORTIE / "rendements_valorisation.csv", index_col=0)
poids_mensuels = pd.read_csv(SORTIE / "poids_mensuels.csv")
derniere = poids_mensuels.date.max()
NOMS = {"P4": "acheteurs", "P5": "vendeurs", "P6": "electricite", "P7": "equipement",
        "P8": "immobilier", "P9": "amont des puces", "P10": "matieres et energie"}
maillon = {t: nom for pf, nom in NOMS.items() for t in appartenance[appartenance.portefeuille == pf].titre}

tableaux = []
for serie in ["P1_reeq", "P1_cons"]:
    w = poids_mensuels[(poids_mensuels.serie == serie) & (poids_mensuels.date == derniere)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[(w > 0) & w.index.isin(rendements.columns)]
    w = w / w.sum()
    bloc = rendements[list(w.index)].iloc[-252:].copy()
    bloc["_marche"] = r_spy.reindex(bloc.index).to_numpy()
    # Covariances par paires, comme a l'etape 3 : un titre recemment cote a moins de 252 seances.
    cov_marche = bloc.cov()["_marche"]
    beta = float(w @ cov_marche[w.index]) / cov_marche["_marche"]
    for nom, poids in [("non couvert", pd.concat([w, pd.Series({"_marche": 0.0})])),
                       ("contrats a terme", pd.concat([w, pd.Series({"_marche": -beta})]))]:
        covariance = bloc.cov().to_numpy() * 252
        x = poids.reindex(bloc.columns).to_numpy()
        sw = covariance @ x
        vol = float(np.sqrt(x @ sw))
        d = pd.DataFrame({"titre": bloc.columns, "contribution": x * sw / vol})
        d["maillon"] = d.titre.map(maillon).fillna("contrat a terme")
        g = d.groupby("maillon").contribution.sum()
        g = (g / vol).rename("part").to_frame()
        g["serie"], g["cas"], g["volatilite"] = serie, nom, vol
        tableaux.append(g.reset_index())
    print(f"{serie} : beta sur 252 seances {beta:.3f}")
par_maillon = pd.concat(tableaux, ignore_index=True)
par_maillon.to_csv(SORTIE / "strategies_residuel_maillon.csv", index=False, encoding="utf-8")
print(par_maillon.pivot_table(index="maillon", columns=["serie", "cas"], values="part").round(3).to_string())
print(par_maillon.groupby(["serie", "cas"]).volatilite.first().round(4).to_string())

P1_reeq : beta sur 252 seances 1.515
P1_cons : beta sur 252 seances 2.297
serie                        P1_cons                      P1_reeq            
cas                 contrats a terme non couvert contrats a terme non couvert
maillon                                                                      
acheteurs                     -0.008       0.030           -0.028       0.015
amont des puces                0.022       0.029            0.140       0.148
contrat a terme               -0.000       0.000            0.000       0.000
electricite                    0.011       0.012            0.034       0.020
equipement                     0.085       0.102            0.146       0.157
immobilier                    -0.001       0.001            0.005       0.012
matieres et energie            0.009       0.012            0.018       0.029
vendeurs                       0.883       0.814            0.684       0.618
serie    cas             
P1_cons  contrats a terme    0.2870
      

## 5. Manifeste

In [8]:
sorties = ["strategies_risque_egal.csv", "strategies_cout.csv", "strategies_risque_residuel.csv",
           "strategies_residuel_maillon.csv"]
(SORTIE / "strategies_manifest.json").write_text(json.dumps({
    "produit_par": "src/comparer_strategies.ipynb", "criteres": "research/plan_projet.md, section VIII",
    "coupure_moities": COUPURE, "bloc": BLOC, "tirages": TIRAGES,
    "entree_sha256": {"couverture_manifest.json": empreinte(SORTIE / "couverture_manifest.json")},
    "sorties_sha256": {s: empreinte(SORTIE / s) for s in sorties}}, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8")
print("ecrit :", ", ".join(sorties))

ecrit : strategies_risque_egal.csv, strategies_cout.csv, strategies_risque_residuel.csv, strategies_residuel_maillon.csv
